In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# load the data
df = pd.read_csv('../data/cleaned/wednesday_cleaned.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (61001, 69)


,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,...,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,Attack
0,443,87261,1,1,0,0,0,0,0.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
1,443,523,2,0,0,0,0,0,0.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
2,53,94304,1,1,56,338,56,56,56.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
3,53,207,2,2,84,246,42,42,42.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
4,123,69022032,2,2,96,96,48,48,48.0,0.0,...,22276.0,0.0,22276,22276,69000000.0,0.0,69000000,69000000,BENIGN,0


In [ ]:
# separate X and y
# Attack is the binary target
X = df.drop(['Label', 'Attack'], axis=1)
y = df['Attack']

print(f'Features: {X.shape}')
print(f'Target: {y.shape}')
print(y.value_counts())

Features: (61001, 67)
Target: (61001,)
Attack
1    56004
0     4997
Name: count, dtype: int64


In [4]:
# split into train and test
# 80% train, 20% test
# stratify to keep class ratio the same in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Train:')
print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print('\nTest:')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

Train:
X_train: (48800, 67)
y_train: (48800,)

Test:
X_test: (12201, 67)
y_test: (12201,)


In [5]:
# make sure both sets have similar attack percentages
print('Train:')
print(y_train.value_counts())
train_attack_pct = (y_train.sum() / len(y_train)) * 100
print(f'{train_attack_pct:.2f}% attacks')

print('\nTest:')
print(y_test.value_counts())
test_attack_pct = (y_test.sum() / len(y_test)) * 100
print(f'{test_attack_pct:.2f}% attacks')

# looks good, both around 91%

Train:
Attack
1    44802
0     3998
Name: count, dtype: int64
91.81% attacks

Test:
Attack
1    11202
0      999
Name: count, dtype: int64
91.81% attacks


In [6]:
# standardize the features
# important: only fit on training data
scaler = StandardScaler()
scaler.fit(X_train)

# now transform both sets using training stats
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('done scaling')
print(X_train_scaled.shape)
print(X_test_scaled.shape)

done scaling
(48800, 67)
(12201, 67)


In [7]:
# check that the mean and std are around 0 and 1
print('train mean:', X_train_scaled.mean(axis=0).mean())
print('train std:', X_train_scaled.std(axis=0).mean())

print('\ntest mean:', X_test_scaled.mean(axis=0).mean())
print('test std:', X_test_scaled.std(axis=0).mean())

train mean: -5.053728688555522e-18
train std: 1.0

test mean: 0.003578225234745424
test std: 0.875324614155965


In [8]:
# convert back to dataframes so we can use column names later
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print(X_train_scaled_df.shape)
print(X_test_scaled_df.shape)
X_train_scaled_df.head()

(48800, 67)
(12201, 67)


,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,...,act_data_pkt_fwd,min_seg_size_forward,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min
46742,-0.117964,1.030309,0.026174,0.006278,-0.005178,0.016720,0.356036,-0.098527,-0.104191,0.251334,...,-0.008792,0.421318,-0.269137,-0.077088,-0.270303,-0.258364,1.143699,-0.187801,1.119083,1.154696
45086,-0.117964,-0.977820,-0.049470,-0.040558,-0.248416,-0.030319,-0.821252,-0.098527,-0.424007,-0.852963,...,-0.021609,0.421318,-0.270396,-0.077088,-0.271495,-0.259649,-0.856346,-0.187801,-0.878047,-0.831812
30604,-0.117964,1.030034,0.038781,0.015645,0.012382,0.016720,0.441027,-0.098527,-0.119203,0.263341,...,-0.008792,0.421318,-0.269770,-0.077088,-0.270903,-0.259011,1.122646,-0.187801,1.098061,1.133785
25896,-0.117964,1.546237,-0.049470,-0.031191,-0.067613,-0.030295,0.053844,-0.098527,0.526890,0.788719,...,-0.008792,0.421318,-0.270396,-0.077088,-0.271495,-0.259649,1.670027,-0.187801,1.644644,1.677461
32168,-0.117964,-0.977796,-0.011648,-0.040558,-0.248416,-0.030319,-0.821252,-0.098527,-0.424007,-0.852963,...,-0.021609,0.421318,-0.270396,-0.077088,-0.271495,-0.259649,-0.856346,-0.187801,-0.878047,-0.831812


12/4/2025 - Umar

did train test split (80/20) with stratify so class balance is same in both sets. got 48800 train samples and 12201 test. attack percentage is around 91.8% in both which is good

applied standardscaler to the features - fitted it only on training data to avoid leakage. then used same scaler to transform test data. checked the means and stds and they look right (train is basically 0 and 1, test is close but not exact)

data is ready for modeling now

### Done with Task

12/4/2025 - Moosa

Created 3 functions to be used across our modeling workflow:

1. fit_model(model, X_train, y_train): Trains any scikit-learn model and returns it.

2. evaluate_model(model, X_test, y_test): Tests the model on unseen data and prints the confusion matrix and classification report.

3. save_model(model, path)

Saves a trained model as a .pkl file so it can be loaded later again.

These functions will be used for models in the training and deployment steps.

In [1]:
from sklearn.metrics import classification_report, confusion_matrix
import pickle

def fit_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    return model

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:\n")
    print(cm)

    report = classification_report(y_test, y_pred)
    print("\nClassification Report:\n")
    print(report)

    return cm, report

def save_model(model, path):
    with open(path, "wb") as f:
        pickle.dump(model, f)

    print(f"Model saved to {path}")

Some context about Pickle for the team:
This is module that converts an object (like a trained machine-learning model) into a binary file so it can be saved and loaded later. Saving a model with pickle will let us use it again during deployment without retraining. When the API starts, just load the .pkl file and make the predictions. Hope this helps!

## Task Complete


In [ ]:
import json

# Get Umar's fitted scaler and extract its parameters
scaler_params = {
    "scaler_type": "StandardScaler",
    "feature_means": scaler.mean_.tolist(),      # Mean for each feature
    "feature_stds": scaler.scale_.tolist(),       # Std for each feature  
    "feature_names": X_train.columns.tolist()     # Feature names in order
}

# Load existing final_features.json 
with open('../data/final_features.json', 'r') as f:
    final_features = json.load(f)

# Add the TRAINING scaler parameters as a new section
final_features['scaler_params'] = scaler_params

# Save it back
with open('../data/final_features.json', 'w') as f:
    json.dump(final_features, f, indent=2)


12/8/2025 - Nafisa - Scaling Parameter Storage

**What I Did:**
- Extracted mean and std from Umar's fitted StandardScaler
- Saved scaler parameters to `../data/final_features.json` under `scaler_params`

**Why:**
- Deployment requires same scaling parameters as training data
- Prevents incorrect predictions from inconsistent scaling

### Done with Task